# Chapter 11 — Information Theory: Surprise, Uncertainty, and the Collider

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/11_information_theory/).

Measure entropy and watch a **collider create mutual information from nothing** — explaining away, in bits.

## Setup

In [ ]:
!pip install genjax -q
print('✅ ready')

## The rain / tea / sign collider

Two independent causes (rain 0.3, tea 0.1) of one effect (the wet-floor sign), via a deterministic OR.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
from genjax import gen, flip

@gen
def rain_tea_sign():
    rain = flip(0.3) @ "rain"
    tea = flip(0.1) @ "tea"
    either = jnp.maximum(rain.astype(int), tea.astype(int))
    sign = flip(either.astype(float)) @ "sign"
    return rain, tea, sign

N = 200000
keys = jr.split(jr.key(0), N)
rain, tea, sign = jax.vmap(lambda k: rain_tea_sign.simulate(k, ()).get_retval())(keys)
rain, tea, sign = rain.astype(int), tea.astype(int), sign.astype(int)

## Entropy and mutual information by Monte Carlo

In [ ]:
def entropy_bits(samples):
    p1 = jnp.clip(jnp.mean(samples), 1e-12, 1 - 1e-12)
    return float(-(p1 * jnp.log2(p1) + (1 - p1) * jnp.log2(1 - p1)))

def mi_bits(a, b, mask=None):
    if mask is not None:
        a, b = a[mask], b[mask]
    mi = 0.0
    for av in (0, 1):
        for bv in (0, 1):
            p_ab = jnp.mean((a == av) & (b == bv))
            p_a = jnp.mean(a == av); p_b = jnp.mean(b == bv)
            if p_ab > 0 and p_a > 0 and p_b > 0:
                mi += float(p_ab * jnp.log2(p_ab / (p_a * p_b)))
    return mi

print(f"H(rain)             = {entropy_bits(rain):.3f} bits")
print(f"H(sign)             = {entropy_bits(sign):.3f} bits")
print(f"I(rain; tea)              = {mi_bits(rain, tea):.3f} bits   (independent)")
print(f"I(rain; tea | sign = 1)   = {mi_bits(rain, tea, sign == 1):.3f} bits   (collider opened)")

## Your turn

1. Compute the entropy of a weighted die (6 with prob 0.5, each of 1–5 with prob 0.1) and compare to a fair die ($\log_2 6 \approx 2.585$ bits).
2. Build a chain $A \to B \to C$ with `flip` and confirm that $I(A;C\mid B)$ drops toward zero — the info-theoretic signature of a *blocked* path.
3. Vary the priors on rain and tea. Does the collider's $I(R;T\mid \text{sign})$ get bigger or smaller as the causes get rarer?